In [ ]:
def load_puv_csv(path):
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df.index.name = "time"
    df = df.rename(columns={"wh_4061": "Hs", "wp_peak": "Tp"})
    df = df.apply(pd.to_numeric, errors="coerce")
    df["Hs"] = df["Hs"].where(df["Hs"] < 99.0)
    df["Tp"] = df["Tp"].where(df["Tp"] < 99.0)
    df["P_kW_m"] = P_CONST * df["Hs"]**2 * df["Tp"]
    return df

puv = load_puv_csv(PUV_FILE)
puv = puv.resample("1h").mean()  # hourly so cumsum = kWh/m
puv["omega"] = puv["Hs"] / (ws_mean * puv["Tp"])

print(f"PUV Data Loaded: {len(puv)} rows")
print(f"  {puv.index.min()} → {puv.index.max()}")
print(f"  ws_mean used for Ω: {ws_mean*100:.3f} cm/s")
puv[["Hs", "Tp", "P_kW_m", "omega"]].describe().round(3)